# Analyse et score — segment sans engagement

## Objectif et périmètre

Ce notebook reconstruit le processus de **risk differentiation** : séparation apprentissage/test, contrôles, discrétisation des variables quantitatives, calcul des WoE/IV, régression logistique et comparaison avec un modèle concurrent. Les seuils, regroupements, références et paramètres sont appris sur le train, puis gelés avant toute évaluation sur le test.

Le test est réservé à l’évaluation finale. Un AUC élevé ne démontre ni la calibration des PD, ni la stabilité temporelle, ni la validité métier des variables. La calibration (LRA, MoC) relève d’une étape distincte.

Les cellules de transformation acceptent des `polars.DataFrame` et les fonctions sont conçues pour être vérifiées sur des données simulées. Les variables qualitatives sont exclues du modèle de ce notebook : leur regroupement doit être spécifié et validé séparément avant de les inclure.

## Corrections méthodologiques intégrées

- Le découpage est stratifié sur la cible ; chaque ligne correspond à un client, selon la confirmation donnée.
- Les seuils d’arbres, profils de risque, références et variables retenues ne sont calculés que sur le train.
- Le test est transformé avec les mêmes seuils et le même encodage que le train.
- L’évaluation du test repose sur les prédictions obtenues à partir de la matrice du test et compare ces probabilités à sa cible.
- Les valeurs manquantes des variables numériques forment une classe explicite. Une classe absente du train mais présente dans le test déclenche une erreur plutôt qu’un encodage inventé.
- Aucun écrêtage automatique des valeurs extrêmes n’est appliqué : sa nécessité doit être justifiée, et ses bornes seraient apprises sur le train uniquement.

## 0. Chargement et paramètres

Cette cellule reprend le chemin et les noms de colonnes présents dans le notebook initial. Elle ne doit être exécutée que dans l’environnement du projet. L’aperçu des lignes est remplacé par l’affichage de la forme et des types.

Vérifier que `ID_COL` identifie bien les clients et que `TARGET` est une cible binaire 0/1. Les variables numériques codées comme catégories doivent être listées dans `COLONNES_CATEGORIELLES_NUMERIQUES`.

In [40]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import statsmodels.api as sm
from scipy.stats import chi2, spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

CACHE = Path("data/segments/mono_sans_engagement/base_mono_sans_engagement.parquet")
ID_COL = "gdt"
TARGET = "ddefaut_ndb"
SEED = 42
TEST_SIZE = 0.20
IV_MIN = 0.02
COLONNES_CATEGORIELLES_NUMERIQUES: set[str] = set()

base = pl.read_parquet(CACHE)
print(f"Dimensions : {base.shape}")
print("Types :", base.schema)

Dimensions : (78118, 187)
Types : Schema({'datdelhis': Date, 'ddefaut_ndb': Float64, 'gdt': String, 'nb_tiers': Float64, 'code_marche_v2': String, 'type_PM_PP': String, 'has_engagement': Float64, 'nb_crt_distincts': Float64, 'engagement': Float64, 'bilan': Float64, 'horsbilan': Float64, 'nb_AC_distincts': Float64, 'nb_agence_gestion_distincts': Float64, 'nb_produits_distincts': Float64, 'nb_credit_immo': Float64, 'nb_credit_immo__sans_garantie': Float64, 'nb_credit_immo__crelog': Float64, 'nb_credit_immo__hypotheque': Float64, 'nb_credit_immo__autre_OCM': Float64, 'nb_credit_immo__contrat_vie': Float64, 'nb_credit_immo__autre_garantie': Float64, 'engagement_immo': Float64, 'engagement_immo__sans_garantie': Float64, 'engagement_immo__crelog': Float64, 'engagement_immo__hypotheque': Float64, 'engagement_immo__autre_OCM': Float64, 'engagement_immo__contrat_vie': Float64, 'engagement_immo__autre_garantie': Float64, 'nb_credit_MLT': Float64, 'nb_credit_MLT__sans_garantie': Float64, 'nb_cred

## 1. Contrat d'entrée et contrôles bloquants

### Rôle de `valider_base`

La fonction vérifie que les colonnes d'identifiant et de cible existent, que chaque identifiant apparaît une seule fois, que la cible n'a ni valeur manquante ni modalité hors de `{0, 1}`, et qu'il existe des prédicteurs numériques utilisables.

Un identifiant dupliqué change l'unité statistique et peut induire une dépendance entre lignes. Une cible invalide ne doit pas être supprimée sans décision explicite. La fonction renvoie uniquement les prédicteurs numériques ; l'identifiant, la cible et les variables numériques déclarées catégorielles sont exclus.

## 2. SÃ©paration apprentissage / test

Les identifiants sont sÃ©parÃ©s de faÃ§on alÃ©atoire et reproductible avec stratification sur la cible. Le contrÃ´le prÃ©alable garantit une ligne par client : il n'y a donc pas de client susceptible d'apparaÃ®tre dans les deux ensembles.

Cette sÃ©paration mesure la performance sur un Ã©chantillon alÃ©atoire tenu Ã  l'Ã©cart. Elle ne remplace pas une validation Out-of-Time. Aucun paramÃ¨tre appris (seuil, regroupement ou coefficient) ne doit Ãªtre choisi d'aprÃ¨s le test.

In [49]:
train_ids, test_ids = train_test_split(
    base[ID_COL].to_list(),
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=base[TARGET].to_list(),
)
train = base.join(pl.DataFrame({ID_COL: train_ids}), on=ID_COL, how="semi")
test = base.join(pl.DataFrame({ID_COL: test_ids}), on=ID_COL, how="semi")


## 3. Valeurs manquantes

### DÃ©composition de `analyser_manquants`

1. Parcourir chaque colonne sans modifier la table.
2. Compter les valeurs nulles et, pour les colonnes flottantes, les `NaN`.
3. Rapporter le nombre, le taux et le nombre de modalitÃ©s distinctes.
4. Calculer le diagnostic sur le train ; le test reste descriptif et ne pilote aucun choix de prÃ©paration.

Les manquants des prÃ©dicteurs numÃ©riques ne sont pas imputÃ©s par une mÃ©diane globale : ils forment une classe dÃ©diÃ©e Ã  la discrÃ©tisation. Les manquants de la cible ou de l'identifiant sont bloquants.

In [50]:
def analyser_manquants(df: pl.DataFrame) -> pl.DataFrame:
    """Retourne le nombre et le taux de null/NaN par colonne."""
    if df.height == 0:
        raise ValueError("Impossible de calculer des taux sur une table vide.")
    resultats = []
    for col, dtype in zip(df.columns, df.dtypes):
        serie = df[col]
        nb_nan = int(serie.is_nan().sum()) if dtype in (pl.Float32, pl.Float64) else 0
        nb = serie.null_count() + nb_nan
        resultats.append({
            "variable": col,
            "nb_manquants": nb,
            "taux_manquants_pct": 100 * nb / df.height,
            "nb_modalites": serie.n_unique(),
        })
    return pl.DataFrame(resultats).sort("taux_manquants_pct", descending=True)

table_manquants_train = analyser_manquants(train)
print(table_manquants_train)

shape: (187, 4)
┌───────────────────────┬──────────────┬────────────────────┬──────────────┐
│ variable              ┆ nb_manquants ┆ taux_manquants_pct ┆ nb_modalites │
│ ---                   ┆ ---          ┆ ---                ┆ ---          │
│ str                   ┆ i64          ┆ f64                ┆ i64          │
╞═══════════════════════╪══════════════╪════════════════════╪══════════════╡
│ DATE_PREMIER_ARR      ┆ 63017        ┆ 100.0              ┆ 1            │
│ AGE_PM_EN_MOIS        ┆ 63017        ┆ 100.0              ┆ 1            │
│ LISTE_CODACVPRO_modif ┆ 63017        ┆ 100.0              ┆ 1            │
│ LISTE_CODNAF2         ┆ 63017        ┆ 100.0              ┆ 1            │
│ LISTE_CODETAJUR       ┆ 63017        ┆ 100.0              ┆ 1            │
│ …                     ┆ …            ┆ …                  ┆ …            │
│ historique_defaut     ┆ 0            ┆ 0.0                ┆ 2            │
│ patrimonial           ┆ 0            ┆ 0.0                

## 4. Diagnostic des valeurs extrÃªmes

### DÃ©composition de `detecter_outliers_iqr`

Pour chaque prÃ©dicteur quantitatif, la fonction calcule Q1, Q3 et l'Ã©cart interquartile sur les valeurs finies du train, puis compte les observations en dehors de `[Q1 - 1,5 Ã— IQR ; Q3 + 1,5 Ã— IQR]`.

Ce critÃ¨re signale des valeurs Ã  examiner ; il ne prouve pas qu'elles sont erronÃ©es. Aucun Ã©crÃªtage n'est effectuÃ© automatiquement : la discrÃ©tisation par arbre peut dÃ©jÃ  isoler une queue de distribution. Si un traitement est retenu, ses bornes devront Ãªtre estimÃ©es sur le train puis rÃ©utilisÃ©es inchangÃ©es sur le test.

In [51]:
def detecter_outliers_iqr(
    df: pl.DataFrame,
    variables: list[str],
) -> pl.DataFrame:
    """Produit un diagnostic IQR sans supprimer ni modifier d'observations."""
    resultats = []
    for col in variables:
        serie = df[col].cast(pl.Float64).drop_nulls()
        serie = serie.filter(serie.is_finite())
        if serie.len() == 0:
            continue
        q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
        iqr = q3 - q1
        bas, haut = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        nb = serie.filter((serie < bas) | (serie > haut)).len()
        resultats.append({
            "variable": col, "Q1": q1, "Q3": q3, "IQR": iqr,
            "borne_inf": bas, "borne_sup": haut, "nb_outliers": nb,
            "taux_outliers_pct": 100 * nb / serie.len(),
        })
    if not resultats:
        return pl.DataFrame(schema={
            "variable": pl.String, "Q1": pl.Float64, "Q3": pl.Float64,
            "IQR": pl.Float64, "borne_inf": pl.Float64, "borne_sup": pl.Float64,
            "nb_outliers": pl.Int64, "taux_outliers_pct": pl.Float64,
        })
    return pl.DataFrame(resultats).sort("taux_outliers_pct", descending=True)

resultats_outliers = detecter_outliers_iqr(train, variables_num)
print(resultats_outliers.head(20))

shape: (20, 8)
┌──────────────┬─────────┬──────────┬──────────┬───────────┬───────────┬─────────────┬─────────────┐
│ variable     ┆ Q1      ┆ Q3       ┆ IQR      ┆ borne_inf ┆ borne_sup ┆ nb_outliers ┆ taux_outlie │
│ ---          ┆ ---     ┆ ---      ┆ ---      ┆ ---       ┆ ---       ┆ ---         ┆ rs_pct      │
│ str          ┆ f64     ┆ f64      ┆ f64      ┆ f64       ┆ f64       ┆ i64         ┆ ---         │
│              ┆         ┆          ┆          ┆           ┆           ┆             ┆ f64         │
╞══════════════╪═════════╪══════════╪══════════╪═══════════╪═══════════╪═════════════╪═════════════╡
│ SOLDE_PENTE_ ┆ -9500.0 ┆ 25833.0  ┆ 35333.0  ┆ -62499.5  ┆ 78832.5   ┆ 16833       ┆ 26.71184    │
│ 3T           ┆         ┆          ┆          ┆           ┆           ┆             ┆             │
│ SOLDE_PENTE_ ┆ -4766.5 ┆ 26733.5  ┆ 31500.0  ┆ -52016.5  ┆ 73983.5   ┆ 16162       ┆ 25.647048   │
│ 4T           ┆         ┆          ┆          ┆           ┆           ┆    

## 5. Associations entre prÃ©dicteurs quantitatifs

### DÃ©composition de `calculer_correlations`

La fonction calcule les corrÃ©lations de Spearman paire par paire sur le train. Pour chaque paire, elle retire seulement les lignes non finies de ces deux variables, calcule le nombre de cas utilisables et renvoie le coefficient et sa p-value.

Une association forte peut signaler une redondance, mais ne suffit ni Ã  prouver une multicolinÃ©aritÃ© problÃ©matique dans le modÃ¨le final, ni Ã  sÃ©lectionner une variable. Les tests multiples et la dÃ©pendance entre variables doivent Ãªtre considÃ©rÃ©s dans l'analyse mÃ©tier.

In [52]:
def calculer_correlations(
    df: pl.DataFrame,
    variables: list[str],
) -> pl.DataFrame:
    """Calcule Spearman par paires avec suppression des cas manquants pairewise."""
    resultats = []
    for var1, var2 in combinations(variables, 2):
        x = df[var1].cast(pl.Float64).fill_null(float("nan")).to_numpy()
        y = df[var2].cast(pl.Float64).fill_null(float("nan")).to_numpy()
        valides = np.isfinite(x) & np.isfinite(y)
        if valides.sum() < 3 or np.unique(x[valides]).size < 2 or np.unique(y[valides]).size < 2:
            continue
        rho, p_value = spearmanr(x[valides], y[valides])
        resultats.append({
            "variable_1": var1, "variable_2": var2,
            "n": int(valides.sum()), "rho_spearman": float(rho),
            "abs_rho": abs(float(rho)), "p_value": float(p_value),
        })
    if not resultats:
        return pl.DataFrame(schema={
            "variable_1": pl.String, "variable_2": pl.String, "n": pl.Int64,
            "rho_spearman": pl.Float64, "abs_rho": pl.Float64, "p_value": pl.Float64,
        })
    return pl.DataFrame(resultats).sort("abs_rho", descending=True)

associations_train = calculer_correlations(train, variables_num)
print(associations_train.filter(pl.col("abs_rho") >= 0.80))

shape: (127, 6)
┌─────────────────────────┬─────────────────────────┬───────┬──────────────┬──────────┬─────────┐
│ variable_1              ┆ variable_2              ┆ n     ┆ rho_spearman ┆ abs_rho  ┆ p_value │
│ ---                     ┆ ---                     ┆ ---   ┆ ---          ┆ ---      ┆ ---     │
│ str                     ┆ str                     ┆ i64   ┆ f64          ┆ f64      ┆ f64     │
╞═════════════════════════╪═════════════════════════╪═══════╪══════════════╪══════════╪═════════╡
│ ANCIENNETE_modif        ┆ anciennete_mois         ┆ 63017 ┆ 1.0          ┆ 1.0      ┆ 0.0     │
│ CRTAD_IND_0205          ┆ CRTAT_IND_0205          ┆ 63017 ┆ 1.0          ┆ 1.0      ┆ 0.0     │
│ NBECR_EVOLUTION_A_vs_A1 ┆ RATIO_ECR               ┆ 49592 ┆ -1.0         ┆ 1.0      ┆ 0.0     │
│ SelectionProb           ┆ SamplingWeight          ┆ 63017 ┆ -1.0         ┆ 1.0      ┆ 0.0     │
│ AGE_PP_EN_MOIS          ┆ age_ans                 ┆ 63017 ┆ 1.0          ┆ 1.0      ┆ 0.0     │
│ … 

## 6. Apprentissage des seuils de discrÃ©tisation

Un arbre univariÃ© propose des coupures supervisÃ©es par la cible. Apprendre ces coupures sur le test ferait intervenir les rÃ©ponses du test dans la construction des variables et rendrait l'Ã©valuation optimiste. La fonction ci-dessous n'accepte donc que le train.

### DÃ©composition de `apprendre_seuils`

1. Exclure les valeurs non finies pour apprendre les coupures ; elles seront conservÃ©es dans une classe manquante sÃ©parÃ©e.
2. Ne lancer l'arbre que si la variable et la cible ont chacune au moins deux valeurs distinctes.
3. Limiter le nombre de feuilles et imposer un effectif minimal par feuille.
4. Retourner les seuils ordonnÃ©s, sans jamais ajuster d'arbre sur le test.

Ces paramÃ¨tres sont des choix de dÃ©part Ã  justifier, et non une optimisation automatique du modÃ¨le final.

In [53]:
def apprendre_seuils(
    df_train: pl.DataFrame,
    variables: list[str],
    target: str,
    max_leaf_nodes: int = 5,
    min_leaf_fraction: float = 0.02,
    random_state: int = 42,
) -> dict[str, list[float]]:
    """Apprend des seuils d'arbre univariÃ© exclusivement sur le train."""
    if not 0 < min_leaf_fraction <= 0.5:
        raise ValueError("min_leaf_fraction doit Ãªtre compris entre 0 et 0,5.")
    seuils = {}
    for col in variables:
        valeurs = df_train[col].cast(pl.Float64).fill_null(float("nan")).to_numpy()
        cible = df_train[target].to_numpy()
        finies = np.isfinite(valeurs)
        x, y = valeurs[finies], cible[finies]
        if np.unique(x).size < 2 or np.unique(y).size < 2:
            seuils[col] = []
            continue
        min_leaf = max(1, int(np.ceil(min_leaf_fraction * len(y))))
        arbre = DecisionTreeClassifier(
            criterion="gini", max_leaf_nodes=max_leaf_nodes,
            min_samples_leaf=min_leaf, random_state=random_state,
        ).fit(x.reshape(-1, 1), y)
        internes = arbre.tree_.children_left != -1
        seuils[col] = sorted(arbre.tree_.threshold[internes].tolist())
    return seuils

seuils_train = apprendre_seuils(train, variables_num, TARGET)
print(f"Variables avec au moins une coupure : {sum(bool(v) for v in seuils_train.values())}")

Variables avec au moins une coupure : 58


## 7. Application des coupures gelÃ©es

### DÃ©composition de `appliquer_binning`

Pour les valeurs finies, `searchsorted` affecte le numÃ©ro d'intervalle dÃ©fini par les coupures apprises sur le train. Le code `-1` est rÃ©servÃ© aux valeurs manquantes ou non finies ; il constitue une modalitÃ© explicite, distincte du premier intervalle.

La mÃªme fonction et le mÃªme dictionnaire `seuils_train` sont utilisÃ©s pour le train et le test. Aucun arbre ni seuil n'est recalculÃ© dans cette Ã©tape.

In [54]:
def appliquer_binning(
    df: pl.DataFrame,
    seuils: dict[str, list[float]],
) -> pl.DataFrame:
    """Transforme les prÃ©dicteurs selon des seuils dÃ©jÃ  appris."""
    expressions = []
    for col, bornes in seuils.items():
        valeurs = df[col].cast(pl.Float64).fill_null(float("nan")).to_numpy()
        classes = np.full(len(valeurs), -1, dtype=np.int64)
        finies = np.isfinite(valeurs)
        classes[finies] = np.searchsorted(
            np.asarray(bornes, dtype=float), valeurs[finies], side="left"
        )
        expressions.append(pl.Series(f"{col}_bin", classes, dtype=pl.Int64))
    return df.with_columns(expressions)

train_binned = appliquer_binning(train, seuils_train)
test_binned = appliquer_binning(test, seuils_train)

## 8. WoE et IV : estimation sur le train

La convention utilisÃ©e est `WoE = ln(part des non-dÃ©fauts / part des dÃ©fauts)`. Un WoE positif correspond donc Ã  une classe dont la part parmi les non-dÃ©fauts dÃ©passe sa part parmi les dÃ©fauts. L'IV est la somme des `(part_non_defauts - part_dÃ©fauts) Ã— WoE`.

Un lissage de Jeffreys (`alpha = 0,5`) Ã©vite les logarithmes infinis. Le bin manquant `-1` est inclus. Les WoE et IV sont estimÃ©s sur le train uniquement ; l'IV sert au tri initial et ne prouve pas un pouvoir prÃ©dictif hors Ã©chantillon.

### DÃ©composition de `calculer_woe_iv`

1. Compter les bons et dÃ©fauts dans chaque bin.
2. Calculer les distributions lissÃ©es par classe de cible.
3. Calculer WoE et contribution Ã  l'IV.
4. Renvoyer le dictionnaire des WoE et le tableau des IV, sans utiliser le test.

In [55]:
def calculer_woe_iv(
    df_train: pl.DataFrame,
    bin_cols: list[str],
    target: str,
    alpha: float = 0.5,
) -> tuple[dict[str, dict[int, float]], pl.DataFrame]:
    """Calcule WoE par classe et IV par variable sur le train."""
    if alpha <= 0:
        raise ValueError("alpha doit Ãªtre strictement positif.")
    mappings, lignes_iv = {}, []
    for col in bin_cols:
        comptes = (
            df_train.select([col, target]).group_by(col)
            .agg(
                (pl.col(target) == 0).sum().alias("nb_bons"),
                (pl.col(target) == 1).sum().alias("nb_defauts"),
            ).sort(col)
        )
        lignes = comptes.to_dicts()
        total_bons = sum(ligne["nb_bons"] for ligne in lignes)
        total_defauts = sum(ligne["nb_defauts"] for ligne in lignes)
        if total_bons == 0 or total_defauts == 0:
            raise ValueError(f"Les deux classes doivent Ãªtre reprÃ©sentÃ©es pour {col!r}.")
        n_bins, mapping, iv = len(lignes), {}, 0.0
        for ligne in lignes:
            p_bons = (ligne["nb_bons"] + alpha) / (total_bons + alpha * n_bins)
            p_defauts = (ligne["nb_defauts"] + alpha) / (total_defauts + alpha * n_bins)
            woe = float(np.log(p_bons / p_defauts))
            iv += (p_bons - p_defauts) * woe
            mapping[int(ligne[col])] = woe
        mappings[col] = mapping
        lignes_iv.append({
            "variable": col, "IV": float(iv), "nb_classes": n_bins,
            "nb_bons": total_bons, "nb_defauts": total_defauts,
        })
    return mappings, pl.DataFrame(lignes_iv).sort("IV", descending=True)

bin_cols = [f"{col}_bin" for col in variables_num]
woe_mappings, iv_train = calculer_woe_iv(train_binned, bin_cols, TARGET)
print(iv_train)

shape: (157, 5)
┌─────────────────────────────┬──────────┬────────────┬─────────┬────────────┐
│ variable                    ┆ IV       ┆ nb_classes ┆ nb_bons ┆ nb_defauts │
│ ---                         ┆ ---      ┆ ---        ┆ ---     ┆ ---        │
│ str                         ┆ f64      ┆ i64        ┆ i64     ┆ i64        │
╞═════════════════════════════╪══════════╪════════════╪═════════╪════════════╡
│ SamplingWeight_bin          ┆ 5.766149 ┆ 2          ┆ 62654   ┆ 363        │
│ anciennete_jour_sain_M0_bin ┆ 0.879195 ┆ 6          ┆ 62654   ┆ 363        │
│ MINSOLDE_PAR_bin            ┆ 0.871873 ┆ 5          ┆ 62654   ┆ 363        │
│ CRTAD_AG_SOLDE_T4_bin       ┆ 0.778259 ┆ 5          ┆ 62654   ┆ 363        │
│ CRTAD_AG_NBJDE_BA_bin       ┆ 0.776848 ┆ 5          ┆ 62654   ┆ 363        │
│ …                           ┆ …        ┆ …          ┆ …       ┆ …          │
│ TOTAL_EPS_PRO_IND_0208_bin  ┆ 0.0      ┆ 1          ┆ 62654   ┆ 363        │
│ NB_CCT_PRO_bin              ┆ 0.0 

## 9. PrÃ©-sÃ©lection des variables

Le seuil `IV_MIN` est fixÃ© avant de consulter le test et reprend le repÃ¨re usuel de 0,02 comme seuil d'information faible. Ce repÃ¨re reste indicatif : il ne remplace ni l'analyse mÃ©tier, ni l'examen de la stabilitÃ©, de la redondance et de la disponibilitÃ© des variables.

Les variables qui passent ce filtre forment une liste de travail. Si aucune variable ne le passe, le notebook s'arrÃªte au lieu de construire artificiellement un modÃ¨le. La sÃ©lection finale doit aussi considÃ©rer la parcimonie et les tests par variable.

In [56]:
variables_retenues = (
    iv_train.filter(pl.col("IV") >= IV_MIN)
    .get_column("variable").to_list()
)
if not variables_retenues:
    raise ValueError(
        f"Aucune variable ne dÃ©passe IV_MIN={IV_MIN}. "
        "RÃ©examiner les bins et la rÃ¨gle de sÃ©lection sur le train."
    )
print(f"Variables retenues pour la modÃ©lisation : {len(variables_retenues)}")

Variables retenues pour la modÃ©lisation : 67


## 10. Encodage indicateur avec rÃ©fÃ©rence de faible risque

### DÃ©composition de `construire_matrices`

Pour chaque variable, les taux de dÃ©faut par bin sont calculÃ©s sur le train. Le bin au taux le plus faible devient la rÃ©fÃ©rence (son indicatrice est omise) ; chaque autre bin, y compris le bin manquant s'il existe dans le train, reÃ§oit une colonne indicatrice.

La liste ordonnÃ©e des colonnes et les groupes de colonnes associÃ©s aux variables sont conservÃ©s. Le test rÃ©utilise strictement cette structure. Si le test contient une classe non observÃ©e au train, une erreur est levÃ©e : aucun effet n'a Ã©tÃ© estimÃ© pour cette classe.

Cette reprÃ©sentation respecte le principe des dummies avec une modalitÃ© de rÃ©fÃ©rence ; elle ne suppose pas que l'Ã©cart de risque entre bins est linÃ©aire.

In [57]:
def construire_matrices(
    train_df: pl.DataFrame,
    test_df: pl.DataFrame,
    variables: list[str],
    target: str,
) -> tuple[np.ndarray, np.ndarray, list[str], dict[str, list[int]], dict[str, int]]:
    """Construit des dummies train/test et choisit la rÃ©fÃ©rence sur le train."""
    colonnes_train, colonnes_test, noms = [], [], []
    groupes, references = {}, {}
    for var in variables:
        bin_col = f"{var}_bin"
        profils = (
            train_df.group_by(bin_col)
            .agg(pl.col(target).mean().alias("taux_defaut"))
            .sort(["taux_defaut", bin_col]).to_dicts()
        )
        if not profils:
            raise ValueError(f"Aucun bin observÃ© pour {var!r} dans le train.")
        categories = [int(ligne[bin_col]) for ligne in profils]
        references[var], groupes[var] = categories[0], []
        bins_train, bins_test = train_df[bin_col].to_numpy(), test_df[bin_col].to_numpy()
        inconnues = set(np.unique(bins_test).tolist()) - set(categories)
        if inconnues:
            raise ValueError(f"Bins non observÃ©s dans le train pour {var!r} : {sorted(inconnues)}")
        for categorie in categories[1:]:
            colonnes_train.append((bins_train == categorie).astype(float))
            colonnes_test.append((bins_test == categorie).astype(float))
            groupes[var].append(len(noms))
            noms.append(f"{var}_bin_{categorie}")
    if not colonnes_train:
        raise ValueError("L'encodage ne contient aucune indicatrice estimable.")
    return (
        np.column_stack(colonnes_train), np.column_stack(colonnes_test),
        noms, groupes, references,
    )

X_train, X_test, noms_dummies, groupes_dummies, references = construire_matrices(
    train_binned, test_binned, variables_retenues, TARGET
)
print(f"Dimensions : train={X_train.shape}, test={X_test.shape}")
print("RÃ©fÃ©rences, dÃ©terminÃ©es sur le train :", references)

ColumnNotFoundError: unable to find column "SamplingWeight_bin_bin"; valid columns: ["datdelhis", "ddefaut_ndb", "gdt", "nb_tiers", "code_marche_v2", "type_PM_PP", "has_engagement", "nb_crt_distincts", "engagement", "bilan", "horsbilan", "nb_AC_distincts", "nb_agence_gestion_distincts", "nb_produits_distincts", "nb_credit_immo", "nb_credit_immo__sans_garantie", "nb_credit_immo__crelog", "nb_credit_immo__hypotheque", "nb_credit_immo__autre_OCM", "nb_credit_immo__contrat_vie", "nb_credit_immo__autre_garantie", "engagement_immo", "engagement_immo__sans_garantie", "engagement_immo__crelog", "engagement_immo__hypotheque", "engagement_immo__autre_OCM", "engagement_immo__contrat_vie", "engagement_immo__autre_garantie", "nb_credit_MLT", "nb_credit_MLT__sans_garantie", "nb_credit_MLT__interfimo", "nb_credit_MLT__autre_OCM", "nb_credit_MLT__PGE", "nb_credit_MLT__autre_garantie", "engagement_MLT", "engagement_MLT__sans_garantie", "engagement_MLT__interfimo", "engagement_MLT__autre_OCM", "engageme

## 11. Estimation de la rÃ©gression logistique

La cible vaut 1 en cas de dÃ©faut. La matrice contient une constante et les indicatrices construites prÃ©cÃ©demment. Les mÃªmes noms et le mÃªme ordre de colonnes sont utilisÃ©s pour train et test.

Le maximum de vraisemblance est estimÃ© sans rÃ©gularisation afin de conserver l'interprÃ©tation des tests statistiques. Si le modÃ¨le ne converge pas, ou si les donnÃ©es induisent une sÃ©paration parfaite, le problÃ¨me doit Ãªtre traitÃ© en revoyant le regroupement/la parcimonie ; l'erreur n'est pas masquÃ©e par un rÃ©sultat de repli.

In [ ]:
y_train = train[TARGET].to_numpy().astype(int)
y_test = test[TARGET].to_numpy().astype(int)
X_train_const = sm.add_constant(X_train, has_constant="add")
X_test_const = sm.add_constant(X_test, has_constant="add")

modele = sm.Logit(y_train, X_train_const).fit(disp=False, maxiter=200)
if not modele.mle_retvals.get("converged", False):
    raise RuntimeError("La rÃ©gression logistique n'a pas convergÃ©.")
print(modele.summary())

## 12. Tests de significativitÃ©

### Rapport de vraisemblance global

Le test global compare le modÃ¨le Ã  un modÃ¨le contenant uniquement une constante. Il indique si l'ensemble des coefficients apporte de l'information statistique ; il ne mesure pas Ã  lui seul la qualitÃ© de l'ordonnancement.

### Rapport de vraisemblance par variable

Pour chaque variable, un modÃ¨le rÃ©duit est rÃ©-estimÃ© aprÃ¨s retrait simultanÃ© de toutes ses indicatrices. La statistique compare les log-vraisemblances des modÃ¨les complet et rÃ©duit, avec un nombre de degrÃ©s de libertÃ© Ã©gal au nombre de coefficients retirÃ©s. Il s'agit d'un test global par variable, pas d'un test de Wald.

Ces tests sont effectuÃ©s sur le train. Avec plusieurs variables, une correction des tests multiples doit Ãªtre considÃ©rÃ©e avant d'interprÃ©ter les p-values.

In [ ]:
LR_global = float(modele.llr)
ddl_global = int(modele.df_model)
p_global = float(modele.llr_pvalue)
print(f"LR global={LR_global:.4f}, ddl={ddl_global}, p-value={p_global:.6g}")


def test_lr_par_variable(
    modele_complet,
    X_sans_constante: np.ndarray,
    y: np.ndarray,
    groupes: dict[str, list[int]],
) -> pl.DataFrame:
    """Compare le modÃ¨le complet Ã  un modÃ¨le privÃ© des dummies d'une variable."""
    resultats = []
    llf_complet = float(modele_complet.llf)
    for variable, indices in groupes.items():
        if not indices:
            resultats.append({
                "variable": variable, "nb_coefficients": 0, "LR": None, "p_value": None,
            })
            continue
        reduit = np.delete(X_sans_constante, indices, axis=1)
        X_reduit = sm.add_constant(reduit, has_constant="add")
        modele_reduit = sm.Logit(y, X_reduit).fit(disp=False, maxiter=200)
        if not modele_reduit.mle_retvals.get("converged", False):
            raise RuntimeError(f"Le modÃ¨le rÃ©duit pour {variable!r} n'a pas convergÃ©.")
        lr = max(0.0, 2 * (llf_complet - float(modele_reduit.llf)))
        ddl = len(indices)
        resultats.append({
            "variable": variable, "nb_coefficients": ddl,
            "LR": lr, "p_value": float(chi2.sf(lr, ddl)),
        })
    return pl.DataFrame(resultats).sort("p_value")

resultats_lr = test_lr_par_variable(modele, X_train, y_train, groupes_dummies)
print(resultats_lr)

## 13. Odds ratios et intervalles de confiance

Les coefficients dÃ©crivent les log-odds relativement aux bins de rÃ©fÃ©rence. L'exponentielle d'un coefficient est un odds ratio ; son intervalle de confiance s'obtient en exponentiant les bornes de l'intervalle du coefficient.

Les odds ratios concernent les catÃ©gories encodÃ©es et le train. Ils ne sont pas des variations de probabilitÃ© ; leur stabilitÃ© et leur cohÃ©rence mÃ©tier doivent Ãªtre examinÃ©es avant toute conclusion.

In [ ]:
noms_parametres = ["const"] + noms_dummies
params = np.asarray(modele.params)
intervalles = np.asarray(modele.conf_int())
resume_or = pl.DataFrame({
    "terme": noms_parametres,
    "coefficient": params,
    "odds_ratio": np.exp(params),
    "OR_IC95_bas": np.exp(intervalles[:, 0]),
    "OR_IC95_haut": np.exp(intervalles[:, 1]),
})
print(resume_or)

## 14. Ã‰valuation de l'ordonnancement

### DÃ©composition de `evaluer_ordonnancement`

- L'AUC rÃ©sume la capacitÃ© Ã  classer un dÃ©faut avant un non-dÃ©faut ; le Gini vaut `2 Ã— AUC - 1`.
- Le KS est le maximum de l'Ã©cart entre les fonctions cumulÃ©es des dÃ©fauts et non-dÃ©fauts (`max(TPR - FPR)` sur la ROC).
- La fonction vÃ©rifie la longueur des vecteurs et exige les deux classes.

Les mÃ©triques du train dÃ©crivent l'ajustement ; celles du test dÃ©crivent la performance hors Ã©chantillon sur le dÃ©coupage choisi. Le seuil de classification n'est pas optimisÃ© car l'objectif ici est l'ordonnancement.

In [ ]:
def evaluer_ordonnancement(
    y: np.ndarray,
    probabilites: np.ndarray,
    nom: str,
) -> dict[str, float | str | int]:
    """Calcule AUC, Gini et KS Ã  partir des probabilitÃ©s de dÃ©faut."""
    y = np.asarray(y, dtype=int)
    probabilites = np.asarray(probabilites, dtype=float)
    if y.shape[0] != probabilites.shape[0]:
        raise ValueError("La cible et les probabilitÃ©s n'ont pas la mÃªme longueur.")
    if np.unique(y).size != 2:
        raise ValueError("Les mÃ©triques ROC exigent les deux classes.")
    fpr, tpr, _ = roc_curve(y, probabilites)
    auc = float(roc_auc_score(y, probabilites))
    return {
        "echantillon": nom, "n": int(y.size), "taux_defaut": float(y.mean()),
        "AUC": auc, "Gini": 2 * auc - 1,
        "KS": float(np.max(tpr - fpr)),
    }

proba_train = np.asarray(modele.predict(X_train_const))
proba_test = np.asarray(modele.predict(X_test_const))
metriques_logit = pl.DataFrame([
    evaluer_ordonnancement(y_train, proba_train, "Train"),
    evaluer_ordonnancement(y_test, proba_test, "Test"),
])
print(metriques_logit)

### Courbe ROC du test

La courbe est tracÃ©e Ã  partir des prÃ©dictions du test (`X_test_const`) et de la cible correspondante (`y_test`). Le test n'est ni refittÃ©, ni rÃ©ordonnÃ© selon ses taux de dÃ©faut. Il ne doit pas servir Ã  retoucher les coupures ou Ã  choisir des variables.

In [ ]:
fpr_test, tpr_test, _ = roc_curve(y_test, proba_test)
auc_test = roc_auc_score(y_test, proba_test)
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_test, tpr_test, label=f"RÃ©gression logistique (AUC = {auc_test:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Classement alÃ©atoire")
ax.set(xlabel="Taux de faux positifs", ylabel="Taux de vrais positifs", title="ROC â€” test")
ax.grid(alpha=0.3)
ax.legend(loc="lower right")
plt.show()

## 15. ModÃ¨le concurrent : forÃªt alÃ©atoire

La rÃ©gression logistique reste le modÃ¨le de rÃ©fÃ©rence interprÃ©table. Une forÃªt alÃ©atoire est ajustÃ©e comme modÃ¨le concurrent sur les mÃªmes variables discrÃ©tisÃ©es et la mÃªme sÃ©paration train/test. Elle peut signaler une relation non linÃ©aire ou des interactions, mais n'est pas retenue automatiquement comme scorecard rÃ©glementaire.

La comparaison utilise les mÃªmes mÃ©triques et le mÃªme test. Les hyperparamÃ¨tres de la forÃªt ne sont pas optimisÃ©s sur ce test ; une recherche de paramÃ¨tres exigerait une validation croisÃ©e interne au train.

In [ ]:
modele_concurrent = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=max(2, int(0.01 * len(y_train))),
    random_state=SEED,
    n_jobs=-1,
)
modele_concurrent.fit(X_train, y_train)
proba_rf_test = modele_concurrent.predict_proba(X_test)[:, 1]
comparaison_test = pl.DataFrame([
    evaluer_ordonnancement(y_test, proba_test, "Logistique"),
    evaluer_ordonnancement(y_test, proba_rf_test, "ForÃªt alÃ©atoire"),
])
print(comparaison_test)

## 16. Conclusion et limites

1. VÃ©rifier convergence, signes/cohÃ©rences des coefficients, effectifs par classe, tests globaux et parcimonie avant de retenir une spÃ©cification.
2. Justifier chaque variable conservÃ©e ou Ã©cartÃ©e au-delÃ  du seul seuil d'IV ; examiner redondance et stabilitÃ© temporelle des classes.
3. Comparer logistique et modÃ¨le concurrent sur les mÃªmes mÃ©triques, sans choisir les paramÃ¨tres Ã  partir du test.
4. La ROC/AUC, le Gini et le KS Ã©valuent la **risk differentiation**. Ils ne calibrent pas une PD : frÃ©quence observÃ©e, LRA, PIT/TTC et MoC relÃ¨vent de la **risk quantification**.
5. La sÃ©paration alÃ©atoire n'est pas une validation temporelle. Si des pÃ©riodes sont disponibles, prÃ©voir un Out-of-Time.
6. Les variables qualitatives brutes ne sont pas intÃ©grÃ©es ici. Leur regroupement, le traitement des modalitÃ©s rares et l'encodage doivent Ãªtre dÃ©finis sans fuite d'information.

### Points Ã  documenter dans le rapport

- Justification de la cible, de l'unitÃ© d'observation et de l'horizon de dÃ©faut.
- Justification des seuils de dÃ©coupage, du minimum d'effectif et des bins retenus.
- ContrÃ´le de monotonie/stabilitÃ© des taux de dÃ©faut, et regroupement expert si nÃ©cessaire.
- RÃ©sultats sur train, test et idÃ©alement Out-of-Time ; aucune dÃ©cision basÃ©e sur une seule mÃ©trique.